# Spatial Processes: Conditioning Non-Gaussian Random Fields

This notebook reproduces the spatial experiments of *LatentFlow: A General Framework for Conditioning Stochastic Processes* (Sharrock, Astfalck & Moss, 2026; Section 6.1, Figure 2 and Appendix G.2). Three spatial priors on an $H \times W$ grid are each conditioned on four kinds of information:

| Prior | Character |
|---|---|
| Gaussian process (Matérn-3/2, lengthscale 0.15) | smooth, light tails |
| Student-$t$ process ($\nu = 4$, Matérn-3/2, lengthscale 0.5) | heavy tails, spatially coupled |
| Cauchy convolution (lengthscale 0.05) | very heavy tails, sharp local features |

| Condition | Data |
|---|---|
| Sparse observations | the field at 30 points |
| Areal maxima | the maximum over each of 8 irregular regions |
| Areal means | the mean over each of 20 irregular regions |
| Censored observations | at 60 points, only whether the field is above ($+$) or below ($-$) a detection limit |

Every prior is the image of Gaussian innovations under a known map: the affine whitening map $m + L\xi$ for the GP, and a non-linear map for the other two. Every condition is a likelihood on the field. The same guided flow handles all twelve combinations. Only two have a closed-form posterior: the linear conditions (sparse observations and areal means) under the GP.

In [ ]:
import sys
from pathlib import Path

import gpytorch
import matplotlib.pyplot as plt
import torch

from flow_inference.conditions import EqualityCondition
from flow_inference.flow_utils import build_grid_in_box
from flow_inference.flows import GaussianIncrementFlow, GaussianProcessFlow
from flow_inference.guidance import GuidanceMC
from flow_inference.maps import CauchyConvolutionMap, StudentTProcessMap
from flow_inference.sampler import IntegratorSampler

sys.path.insert(0, str(Path.cwd().parent))  # examples/, for plotting.py
from plotting import shared_range, show_overlay

torch.set_default_dtype(torch.float64)

In [ ]:
SEED = 0
torch.manual_seed(SEED)

H, W = 32, 32  # grid size
num_samples = 5  # posterior samples per prior and condition
device = "cuda" if torch.cuda.is_available() else "cpu"

NUM_SENSORS = 30  # sparse point observations
NUM_MAX_REGIONS = 8  # areal maxima
NUM_MEAN_REGIONS = 20  # areal means
NUM_CENSORED = 60  # censored observations

## 1. Spatial priors

- **Gaussian process**: `GaussianProcessFlow` diffuses in the whitened space of the GP, so the generator is the affine map $f = L\xi$ with $LL^\top = K$.
- **Student-$t$ process**: a GP divided by a shared $\sqrt{V/\nu}$ with $V \sim \chi^2_\nu$, giving $t_\nu$ marginals (`StudentTProcessMap`).
- **Cauchy convolution**: a Cauchy noise source convolved with a short-lengthscale kernel (`CauchyConvolutionMap`). Its infinite-variance source produces sharp, highly localised features.

The non-Gaussian priors are maps from Gaussian increments to a field, wrapped in a `GaussianIncrementFlow`. Each prior captures the kernel when it is built, so one kernel object is reused with a different lengthscale per prior.

In [ ]:
coords = build_grid_in_box(H, W, device=device)  # (H*W, 2)
kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.MaternKernel(nu=1.5)).to(device)
kernel.outputscale = 1.0

kernel.base_kernel.lengthscale = 0.15
gp_flow = GaussianProcessFlow(kernel, gpytorch.means.ZeroMean(), coords, output_shape=(H, W), device=device)

kernel.base_kernel.lengthscale = 0.5
student_t_flow = GaussianIncrementFlow(StudentTProcessMap(H=H, W=W, kernel=kernel, nu=4, device=device), device=device)

kernel.base_kernel.lengthscale = 0.05
cauchy_flow = GaussianIncrementFlow(
    CauchyConvolutionMap(H=H, W=W, kernel=kernel, grid_fidelity_ratio=1, device=device), device=device
)

flows = {
    "Gaussian process": gp_flow,
    "Student-t process": student_t_flow,
    "Cauchy convolution": cauchy_flow,
}

## 2. Sampling helpers

`prior_samples` integrates the probability-flow ODE with no guidance; `posterior_samples` adds Monte Carlo guidance from a condition and integrates the reverse SDE. Each demo below draws a ground-truth field from the prior, observes it, and draws posterior samples consistent with the observation. `results` collects every posterior for the summary figure.

In [ ]:
def prior_samples(flow, num_steps=100):
    return IntegratorSampler(flow=flow).sample(num_samples, num_steps=num_steps).detach()


def posterior_samples(flow, condition, num_steps=500, max_norm=1e2):
    guidance = GuidanceMC(flow, condition, num_mc_samples=1, max_norm=max_norm).to(device)
    return IntegratorSampler(flow=flow).sample(num_samples, guidance=guidance, num_steps=num_steps, sde=True).detach()


def draw_ground_truth(flow):
    """One prior draw, (1, 1, H*W): the generator applied to fresh Gaussian innovations."""
    xi = torch.randn((1, flow.channels, flow.latent_dim), device=device)
    return flow._to_data_space(xi).detach()


def show(fields, **kwargs):
    show_overlay(fields, h=H, w=W, n=num_samples, **kwargs)


results = {name: {} for name in flows}
truths = {name: draw_ground_truth(flow) for name, flow in flows.items()}

## 3. Prior samples

Unconditioned samples from each prior: the smooth Gaussian field, the Student-$t$ field whose overall scale varies from draw to draw, and the Cauchy field with its isolated extreme spikes.

In [ ]:
for name, flow in flows.items():
    results[name]["Prior"] = prior_samples(flow)
    print(name)
    show(results[name]["Prior"])

## 4. Observation layouts

The sensor locations and regions are shared across priors, so the rows of the summary figure are directly comparable. Regions are Voronoi cells of random seed pixels.

In [ ]:
def voronoi_regions(num_regions):
    """(H*W,) region index of every pixel, for the Voronoi cells of `num_regions` random seed pixels."""
    seeds = coords[torch.randperm(H * W, device=device)[:num_regions]]
    return torch.cdist(coords, seeds).argmin(dim=1)


sensor_locs = torch.randperm(H * W, device=device)[:NUM_SENSORS]
max_regions = voronoi_regions(NUM_MAX_REGIONS)
mean_regions = voronoi_regions(NUM_MEAN_REGIONS)
censored_locs = torch.randperm(H * W, device=device)[:NUM_CENSORED]

## 5. Sparse observations

Observe the field at 30 points and recover the whole field: kriging, but for any prior. `PointObservation` is an `EqualityCondition` on the observed pixels.

In [ ]:
class PointObservation(EqualityCondition):
    """The field at flat pixel indices `locs` equals `values`."""

    def __init__(self, locs, values, sigma=1e-3):
        super().__init__(sigma=sigma)
        self.register_buffer("locs", locs.long())  # (A,)
        self.register_buffer("values", values)  # (A,)

    def residual(self, x):
        x = x.reshape(x.shape[0], -1)  # (B, N)
        return (x[:, self.locs] - self.values[None]).unsqueeze(1)  # (B, 1, A)


sensor_markers = [{"flat_idx": sensor_locs, "marker": "o", "color": "white", "edgecolor": "black",
                   "s": 18, "label": f"{NUM_SENSORS} observations"}]
for name, flow in flows.items():
    truth = truths[name].reshape(-1)
    condition = PointObservation(sensor_locs, truth[sensor_locs], sigma=1e-5).to(device)
    post = results[name]["Sparse observations"] = posterior_samples(flow, condition)

    vmin, vmax = shared_range(truths[name], post)
    print(f"{name}: truth | posterior samples (they must pass through the observed points)")
    show(truths[name], titles=["ground truth"], markers=sensor_markers, vmin=vmin, vmax=vmax)
    show(post, markers=sensor_markers, vmin=vmin, vmax=vmax)

## 6. Areal maxima

Each of 8 regions reports only its **maximum** (e.g. the peak pollutant reading in a monitoring zone): a non-linear operator that constrains the extreme of each region rather than its bulk. We encode it as two soft constraints per region: no pixel may exceed the reported maximum, and a smooth log-sum-exp surrogate of the region's maximum must attain it (the hard `max` has zero gradient away from the argmax).

In [ ]:
class RegionMaxCondition(EqualityCondition):
    """Match the maximum of the field within each of R regions. Residual (length 2R):
      over-shoot: sum over each region of max(0, x - target_max)
      attainment: logsumexp_beta(x_region) / beta - target_max
    Larger beta gives a sharper surrogate and stiffer gradients.
    """

    def __init__(self, region_ids, region_maxes, beta=12.0, sigma=1e-2):
        super().__init__(sigma=sigma)
        self.R = int(region_maxes.numel())
        self.beta = beta
        self.register_buffer("region_ids", region_ids.reshape(-1).long())  # (N,) in {0..R-1}
        self.register_buffer("region_maxes", region_maxes.reshape(-1))  # (R,)

    def _region_softmax(self, x):
        # Per-region logsumexp / beta, stabilised by the per-region max.
        B = x.shape[0]
        ids = self.region_ids[None].expand(B, -1)  # (B, N)
        neg_inf = torch.full((B, self.R), float("-inf"), device=x.device, dtype=x.dtype)
        m = neg_inf.scatter_reduce(1, ids, x, reduce="amax", include_self=True)  # (B, R)
        m_pix = m.gather(1, ids)  # (B, N)
        z = torch.zeros(B, self.R, device=x.device, dtype=x.dtype)
        sumexp = z.scatter_add(1, ids, torch.exp(self.beta * (x - m_pix)))  # (B, R)
        return m + torch.log(sumexp) / self.beta  # (B, R)

    def residual(self, x):
        B = x.shape[0]
        x = x.reshape(B, -1)  # (B, N)
        tgt_pix = self.region_maxes[self.region_ids][None]  # (1, N)

        overshoot = torch.clamp(x - tgt_pix, min=0.0)  # (B, N)
        z = torch.zeros(B, self.R, device=x.device, dtype=x.dtype)
        over_r = z.scatter_add(1, self.region_ids[None].expand(B, -1), overshoot)  # (B, R)

        attain = self._region_softmax(x) - self.region_maxes[None]  # (B, R)
        return torch.cat([over_r, attain], dim=1).unsqueeze(1)  # (B, 1, 2R)


for name, flow in flows.items():
    truth = truths[name].reshape(-1)
    region_maxes = torch.full((NUM_MAX_REGIONS,), float("-inf"), device=device, dtype=truth.dtype) \
        .scatter_reduce(0, max_regions, truth, reduce="amax", include_self=True)
    condition = RegionMaxCondition(max_regions, region_maxes, beta=1000.0, sigma=1e-5).to(device)
    post = results[name]["Areal max"] = posterior_samples(flow, condition)

    observed = region_maxes[max_regions].view(1, H, W)
    vmin, vmax = shared_range(truths[name], observed, post)
    print(f"{name}: truth | observed region maxima (the only data) | posterior samples")
    show(torch.cat([truths[name].reshape(1, H, W), observed]), titles=["ground truth", "region maxima"],
         boundaries=max_regions, vmin=vmin, vmax=vmax)
    show(post, boundaries=max_regions, vmin=vmin, vmax=vmax)

## 7. Areal means

Each of 20 regions reports only its **mean** (think census tracts or satellite footprints), and we recover a plausible fine-scale field. `RegionMeanCondition` is a linear operator, so under the GP prior alone this posterior is Gaussian; under the other two it is not.

In [ ]:
class RegionMeanCondition(EqualityCondition):
    """Match the mean of the field within each of R regions."""

    def __init__(self, region_ids, region_means, sigma=1e-2):
        super().__init__(sigma=sigma)
        self.R = int(region_means.numel())
        ids = region_ids.reshape(-1).long()
        self.register_buffer("region_ids", ids)  # (N,) in {0..R-1}
        self.register_buffer("region_means", region_means.reshape(-1))  # (R,)
        counts = torch.zeros(self.R, device=ids.device).index_add(0, ids, torch.ones(ids.numel(), device=ids.device))
        self.register_buffer("counts", counts.clamp(min=1))  # (R,)

    def residual(self, x):
        B = x.shape[0]
        x = x.reshape(B, -1)  # (B, N)
        sums = torch.zeros(B, self.R, device=x.device, dtype=x.dtype).index_add(1, self.region_ids, x)
        return (sums / self.counts - self.region_means[None]).unsqueeze(1)  # (B, 1, R)


ones = torch.ones(H * W, device=device)
mean_counts = torch.zeros(NUM_MEAN_REGIONS, device=device).index_add(0, mean_regions, ones).clamp(min=1)
for name, flow in flows.items():
    truth = truths[name].reshape(-1)
    region_means = torch.zeros(NUM_MEAN_REGIONS, device=device).index_add(0, mean_regions, truth) / mean_counts
    condition = RegionMeanCondition(mean_regions, region_means, sigma=1e-5).to(device)
    post = results[name]["Areal mean"] = posterior_samples(flow, condition)

    observed = region_means[mean_regions].view(1, H, W)
    vmin, vmax = shared_range(truths[name], observed, post)
    print(f"{name}: truth | observed region means (the only data) | posterior samples")
    show(torch.cat([truths[name].reshape(1, H, W), observed]), titles=["ground truth", "region means"],
         boundaries=mean_regions, vmin=vmin, vmax=vmax)
    show(post, boundaries=mean_regions, vmin=vmin, vmax=vmax)

## 8. Censored observations

At 60 points we learn only whether the field is above ($+$) or below ($-$) a detection limit, as with a sensor that reports exceedance of a threshold but not the value. The limit is the median of the true field, so each prior gets a balanced mix of $+$ and $-$. The correct posterior integrates over the censored half-space rather than substituting the limit; a hinge residual that is zero once the value is beyond the limit by a margin encodes exactly this.

In [ ]:
class CensoredObservationCondition(EqualityCondition):
    """Z(s_i) is known only to be below (left-censored) or above (right-censored) `limit`.
    Hinge residual, zero once the value is beyond the limit by `margin`:
      below: max(0, Z - (limit - margin))     above: max(0, (limit + margin) - Z)
    """

    def __init__(self, locations, below, limit=0.0, sigma=1e-2, margin=0.2):
        super().__init__(sigma=sigma)
        self.limit, self.margin = limit, margin
        self.register_buffer("locs", locations.long())  # (A,) flat indices
        self.register_buffer("below", below.bool())  # (A,)

    def residual(self, x):
        vals = x.reshape(x.shape[0], -1)[:, self.locs]  # (B, A)
        viol_below = torch.clamp(vals - (self.limit - self.margin), min=0.0)
        viol_above = torch.clamp((self.limit + self.margin) - vals, min=0.0)
        return torch.where(self.below[None, :], viol_below, viol_above).unsqueeze(1)  # (B, 1, A)


def censor_markers(below):
    return [
        {"flat_idx": censored_locs[~below], "marker": "+", "color": "red", "s": 40, "label": "above the limit (+)"},
        {"flat_idx": censored_locs[below], "marker": "_", "color": "deepskyblue", "s": 40, "label": "below the limit (-)"},
    ]


censored_below = {}
for name, flow in flows.items():
    truth = truths[name].reshape(-1)
    limit = truth.median().item()
    below = censored_below[name] = truth[censored_locs] < limit
    condition = CensoredObservationCondition(censored_locs, below, limit=limit, sigma=1e-5).to(device)
    post = results[name]["Censored observations"] = posterior_samples(flow, condition)

    signs_match = ((post.reshape(num_samples, -1)[:, censored_locs] < limit) == below[None]).double().mean().item()
    vmin, vmax = shared_range(truths[name], post)
    print(f"{name}: truth | posterior samples ({signs_match:.0%} of censored signs reproduced)")
    show(truths[name], titles=["ground truth"], markers=censor_markers(below), vmin=vmin, vmax=vmax)
    show(post, markers=censor_markers(below), vmin=vmin, vmax=vmax)

## 9. Summary

One posterior sample per prior (rows) and condition (columns), as in Figure 2 of the LatentFlow paper. The samples respect each condition while keeping the character of their prior: smooth for the GP, globally rescaled for the Student-$t$, spiky for the Cauchy field.

In [ ]:
def draw_boundaries(ax, region_ids):
    """White lines between pixels in different regions."""
    b = region_ids.reshape(H, W).cpu()
    for r, c in (b[:, :-1] != b[:, 1:]).nonzero().tolist():
        ax.plot([c + 0.5, c + 0.5], [r - 0.5, r + 0.5], color="white", lw=0.8)
    for r, c in (b[:-1, :] != b[1:, :]).nonzero().tolist():
        ax.plot([c - 0.5, c + 0.5], [r + 0.5, r + 0.5], color="white", lw=0.8)


def draw_points(ax, locs, **style):
    locs = locs.cpu()
    ax.scatter(locs % W, locs // W, **style)


columns = ["Prior", "Sparse observations", "Areal max", "Areal mean", "Censored observations"]
fig, axes = plt.subplots(len(flows), len(columns), figsize=(2.6 * len(columns), 2.6 * len(flows)))
for row, name in enumerate(flows):
    fields = [results[name][column][0].reshape(H, W) for column in columns]
    vmin, vmax = shared_range(*fields)
    for col, (column, field) in enumerate(zip(columns, fields)):
        ax = axes[row, col]
        ax.imshow(field.cpu(), cmap="viridis", vmin=vmin, vmax=vmax)
        ax.set_xticks([])
        ax.set_yticks([])
        if column == "Sparse observations":
            draw_points(ax, sensor_locs, s=14, facecolors="white", edgecolors="black", linewidths=0.6)
        elif column == "Areal max":
            draw_boundaries(ax, max_regions)
        elif column == "Areal mean":
            draw_boundaries(ax, mean_regions)
        elif column == "Censored observations":
            below = censored_below[name]
            draw_points(ax, censored_locs[~below], marker="+", s=30, color="red")
            draw_points(ax, censored_locs[below], marker="_", s=30, color="deepskyblue")
        if row == 0:
            ax.set_title(column, fontsize=10)
        if col == 0:
            ax.set_ylabel(name, fontsize=10)
plt.tight_layout()